In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from flask import Flask, request, jsonify

data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test, check_additivity=False)

shap.summary_plot(shap_values, X_test)

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test.iloc[0],
        feature_names=X_test.columns
    )
)

shap.initjs()
force_plot = shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test.iloc[0]
)

shap.save_html("shap_force_plot.html", force_plot)

app = Flask(__name__)

@app.route('/')
def home():
    return "ML Model API is running. Use /predict endpoint."

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    features = np.array(data["data"]).reshape(1, -1)
    prediction = model.predict(features)
    return jsonify({
        "prediction": float(prediction[0])
    })

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Mar/2026 23:12:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Mar/2026 23:12:21] "GET /favicon.ico HTTP/1.1" 404 -
